# PrecisionPilot — 01 · Fine-tune ResNet-18 on CIFAR-10 (224 px)

**Before running:** *Settings → Accelerator → GPU T4* and *Settings → Internet → On*.
Then *Run All*. The run takes roughly 10 minutes on a T4.

What it does:
1. clones the PrecisionPilot repo and installs it,
2. checks the environment,
3. downloads CIFAR-10 and loads the committed data splits (`data/splits.json`),
4. fine-tunes the ImageNet-pretrained ResNet-18 (AMP, SGD + cosine, 8 epochs),
5. reports FP32 / all-FP16 top-1 on the validation and test splits,
6. copies `resnet18_cifar10.pt` (+ `.json` training log) to `/kaggle/working/` for download.

**After it finishes:** open the *Output* panel, download `resnet18_cifar10.pt` and
`resnet18_cifar10.json`, and place both in `checkpoints/` of your local repo
(Mac and Windows). Also download `baselines_t4.json` into `results/baselines/`.

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")
if not os.path.exists("PrecisionPilot"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/rajat-gondkar/PrecisionPilot.git"], check=True)
os.chdir("/kaggle/working/PrecisionPilot")
subprocess.run(["git", "pull", "--ff-only"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!python -m precisionpilot env-check

### Data splits
The split file is committed in the repo, so every machine uses identical indices. This cell downloads CIFAR-10 and verifies the splits.

In [ ]:
!python -m precisionpilot make-splits

### Fine-tune
Override any setting with `--set`, e.g. `--set train.epochs=10`.

In [ ]:
!python -m precisionpilot train --device cuda

### Baselines (FP32 and all-FP16)

In [ ]:
!python -m precisionpilot evaluate --device cuda

In [ ]:
import shutil, glob
for f in ["checkpoints/resnet18_cifar10.pt", "checkpoints/resnet18_cifar10.json",
          *glob.glob("results/baselines/*.json")]:
    shutil.copy(f, "/kaggle/working/")
    print("ready for download:", "/kaggle/working/" + f.split("/")[-1])